# E5  :  Fine-tuning MTEB French

**Objectif :** Fine-tuner `intfloat/multilingual-e5-base` sur un mélange français retrieval + STS + paraphrases + NLI, avec replay multilingue.  
**Hardware cible :** RTX 3070 (8 GB VRAM)  
**Approche :** Sentence Transformers + PEFT LoRA, split anti-fuite, pertes multi-tâches, sélection retrieval/STS et contrôles gradients/poids/rechargement.  
**Références :** [E5 multilingual](https://huggingface.co/intfloat/multilingual-e5-base) · [Training multi-dataset Sentence Transformers](https://sbert.net/docs/sentence_transformer/training_overview.html#multi-dataset-training)

## 0. Imports & vérifications

In [1]:
# Cellule 0 — redémarrer le kernel puis exécuter ceci en premier

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", device)

if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("vrAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

torch: 2.12.1+cu130
cuda: True
device: cuda
gpu: NVIDIA GeForce RTX 3070
vrAM GB: 8.0


In [2]:
# Cellule 1 — imports

from datasets import concatenate_datasets, load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import (
    CachedMultipleNegativesRankingLoss,
    CoSENTLoss,
)
from sentence_transformers.sentence_transformer.evaluation import (
    EmbeddingSimilarityEvaluator,
    InformationRetrievalEvaluator,
    SequentialEvaluator,
    SimilarityFunction,
)
from sentence_transformers.base.sampler import BatchSamplers, MultiDatasetBatchSamplers
from sentence_transformers.util import cos_sim
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import TrainerCallback

print("Imports OK")

W0707 23:03:28.891000 10444 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0707 23:03:28.981000 10444 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Imports OK


In [3]:
# Cellule 2 — configuration

MODEL_NAME = "intfloat/multilingual-e5-base"

OUTPUT_DIR = "./e5-fr-miracl-lora"
MAX_SEQ_LENGTH = 256

TRAIN_SIZE = 50_000  # MIRACL-fr
NLI_TRAIN_SIZE = 18_000
STS_TRAIN_SIZE = 12_000
PARAPHRASE_TRAIN_SIZE = 4_000
REPLAY_PER_LANGUAGE = 2_000
REPLAY_CONFIGS = ("en-fr", "en-de", "en-es", "en-ar")
EVAL_QUERY_SIZE = 200
EVAL_NEGATIVES_PER_QUERY = 8
STS_EVAL_SIZE = 2_000

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64

NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.10

assert TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS == 64

print({
    "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "miracl_train_size": TRAIN_SIZE,
    "nli_train_size": NLI_TRAIN_SIZE,
    "sts_train_size": STS_TRAIN_SIZE,
    "paraphrase_train_size": PARAPHRASE_TRAIN_SIZE,
    "multilingual_replay_size": REPLAY_PER_LANGUAGE * len(REPLAY_CONFIGS),
    "heldout_queries": EVAL_QUERY_SIZE,
    "epochs": NUM_EPOCHS,
    "lr": LEARNING_RATE,
})

{'effective_batch_size': 64, 'miracl_train_size': 50000, 'nli_train_size': 18000, 'sts_train_size': 12000, 'paraphrase_train_size': 4000, 'multilingual_replay_size': 8000, 'heldout_queries': 200, 'epochs': 1, 'lr': 5e-05}


In [4]:
# Cellule 3 — chargement modèle baseline

model = SentenceTransformer(
    MODEL_NAME,
    device=device,
)

model.max_seq_length = MAX_SEQ_LENGTH

print("embedding dim:", model.get_embedding_dimension())
print("max_seq_length:", model.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
f:\codeAI\llm\.venv\lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


embedding dim: 768
max_seq_length: 256


In [5]:
# Cellule 4 — test obligatoire de convention E5

queries = [
    "query: Quelle est la capitale de la France ?",
    "query: Quel animal miaule ?",
]

passages = [
    "passage: Paris est la capitale de la France.",
    "passage: Le chat est un animal domestique qui miaule.",
    "passage: Le moteur d'une voiture transforme de l'énergie en mouvement.",
]

q_emb = model.encode(queries, normalize_embeddings=True)
p_emb = model.encode(passages, normalize_embeddings=True)

scores = q_emb @ p_emb.T

print(scores)

assert scores[0, 0] > scores[0, 1]
assert scores[1, 1] > scores[1, 2]

print("Convention query/passage OK")

[[0.8838406  0.70628965 0.7135833 ]
 [0.7198016  0.82489645 0.73087937]]
Convention query/passage OK


In [6]:
# Cellule 5 — chargement des sources (uniquement leurs splits train/validation)

ds = load_dataset(
    "sentence-transformers/miracl", "fr-triplet-all", split="train"
)
fr_sts_splits = load_dataset("CATIE-AQ/frenchSTS")
fr_nli_raw = load_dataset("CATIE-AQ/frenchNLI", split="train")
replay_raw = {
    config: load_dataset(
        "sentence-transformers/parallel-sentences-global-voices",
        config,
        split=f"train[:{REPLAY_PER_LANGUAGE}]",
    )
    for config in REPLAY_CONFIGS
}

print({
    "miracl_rows": len(ds),
    "french_sts_train_rows": len(fr_sts_splits["train"]),
    "french_nli_rows": len(fr_nli_raw),
    "replay_configs": list(replay_raw),
})

{'miracl_rows': 232100, 'french_sts_train_rows': 12227, 'french_nli_rows': 555798, 'replay_configs': ['en-fr', 'en-de', 'en-es', 'en-ar']}


In [7]:
# Cellule 6 — split MIRACL groupé + préparation multi-dataset sans fuite

anchor_column = ds["anchor"]
positive_column = ds["positive"]
negative_column = ds["negative"]
unique_anchors = np.array(sorted(set(anchor_column)), dtype=object)
split_rng = np.random.default_rng(SEED)
split_rng.shuffle(unique_anchors)

assert EVAL_QUERY_SIZE < len(unique_anchors)
eval_anchor_set = set(unique_anchors[:EVAL_QUERY_SIZE].tolist())
eval_indices = [i for i, anchor in enumerate(anchor_column) if anchor in eval_anchor_set]
eval_ds = ds.select(eval_indices)
eval_positive_texts = set(eval_ds["positive"])
heldout_texts = eval_anchor_set | eval_positive_texts

# Aucun positif du holdout ne peut apparaître dans MIRACL train, même comme négatif.
train_candidate_indices = [
    i
    for i, (anchor, positive, negative) in enumerate(
        zip(anchor_column, positive_column, negative_column)
    )
    if anchor not in eval_anchor_set
    and positive not in eval_positive_texts
    and negative not in eval_positive_texts
]
split_rng.shuffle(train_candidate_indices)
assert len(train_candidate_indices) >= TRAIN_SIZE
train_ds = ds.select(train_candidate_indices[:TRAIN_SIZE]).select_columns(
    ["anchor", "positive", "negative"]
)

def remove_heldout_texts(dataset, text_columns, description):
    return dataset.filter(
        lambda row: all(row[column] not in heldout_texts for column in text_columns),
        desc=description,
    )

# STS français scoré -> CoSENT. OpusParcus est exclu pour rester sur le chemin CC-BY.
sts_source = fr_sts_splits["train"].filter(
    lambda row: row["dataset"] != "opusparcus", desc="Filtrage licence STS"
)
sts_source = remove_heldout_texts(
    sts_source, ["sentence1", "sentence2"], "Anti-leak FrenchSTS"
).shuffle(seed=SEED)
sts_train = sts_source.select(range(min(STS_TRAIN_SIZE, len(sts_source)))).select_columns(
    ["sentence1", "sentence2", "score"]
)

# Paires paraphrastiques fortes, entraînées séparément avec MNRL.
paraphrase_source = sts_source.filter(
    lambda row: row["score"] >= 0.80, desc="Sélection paraphrases FR"
).shuffle(seed=SEED + 1)
paraphrase_train = paraphrase_source.select(
    range(min(PARAPHRASE_TRAIN_SIZE, len(paraphrase_source)))
).select_columns(["sentence1", "sentence2"]).rename_columns(
    {"sentence1": "anchor", "sentence2": "positive"}
)

# NLI français: implication=1, neutre=0.5, contradiction=0 pour CoSENT.
nli_source = remove_heldout_texts(
    fr_nli_raw, ["premise", "hypothesis"], "Anti-leak FrenchNLI"
)
nli_per_label = NLI_TRAIN_SIZE // 3
nli_parts = []
for label in (0, 1, 2):
    label_dataset = nli_source.filter(
        lambda row, expected=label: int(row["label"]) == expected,
        desc=f"FrenchNLI label {label}",
    ).shuffle(seed=SEED + label)
    assert len(label_dataset) >= nli_per_label
    nli_parts.append(label_dataset.select(range(nli_per_label)))
nli_source = concatenate_datasets(nli_parts).shuffle(seed=SEED)
nli_train = nli_source.map(
    lambda row: {
        "sentence1": row["premise"],
        "sentence2": row["hypothesis"],
        "score": {0: 1.0, 1: 0.5, 2: 0.0}[int(row["label"])],
    },
    remove_columns=nli_source.column_names,
    desc="Conversion NLI vers similarité",
)

# Petit replay multilingue pour limiter l'oubli catastrophique.
replay_parts = []
for config, replay_dataset in replay_raw.items():
    replay_dataset = remove_heldout_texts(
        replay_dataset, ["english", "non_english"], f"Anti-leak replay {config}"
    ).select_columns(["english", "non_english"]).rename_columns(
        {"english": "anchor", "non_english": "positive"}
    )
    replay_parts.append(replay_dataset)
replay_train = concatenate_datasets(replay_parts).shuffle(seed=SEED)

train_datasets = {
    "miracl": train_ds,
    "fr_sts": sts_train,
    "fr_paraphrase": paraphrase_train,
    "fr_nli": nli_train,
    "multilingual_replay": replay_train,
}

# Audit strict: aucune requête/aucun positif held-out dans aucune source de train.
training_text_columns = {
    "miracl": ["anchor", "positive", "negative"],
    "fr_sts": ["sentence1", "sentence2"],
    "fr_paraphrase": ["anchor", "positive"],
    "fr_nli": ["sentence1", "sentence2"],
    "multilingual_replay": ["anchor", "positive"],
}
all_training_texts = set()
for dataset_name, dataset in train_datasets.items():
    for column in training_text_columns[dataset_name]:
        all_training_texts.update(dataset[column])

text_overlap = all_training_texts & heldout_texts
assert not text_overlap, f"Data leak multi-dataset: {len(text_overlap)} textes"
assert set(train_ds["anchor"]).isdisjoint(eval_anchor_set)

print({name: len(dataset) for name, dataset in train_datasets.items()})
print({
    "eval_unique_queries": len(eval_anchor_set),
    "multi_dataset_text_overlap": len(text_overlap),
})
print("Préparation multi-dataset anti-leak OK")

FrenchNLI label 0:   0%|          | 0/555798 [00:00<?, ? examples/s]

FrenchNLI label 1:   0%|          | 0/555798 [00:00<?, ? examples/s]

FrenchNLI label 2:   0%|          | 0/555798 [00:00<?, ? examples/s]

Conversion NLI vers similarité:   0%|          | 0/18000 [00:00<?, ? examples/s]

{'miracl': 50000, 'fr_sts': 12000, 'fr_paraphrase': 3822, 'fr_nli': 18000, 'multilingual_replay': 8000}
{'eval_unique_queries': 200, 'multi_dataset_text_overlap': 0}
Préparation multi-dataset anti-leak OK


In [8]:
# Cellule 7 — évaluateur combiné retrieval + STS, entièrement held-out

def build_ir_evaluator(
    dataset,
    training_texts,
    negatives_per_query=8,
    name="miracl-fr-heldout",
):
    grouped = {}
    for row in dataset:
        group = grouped.setdefault(row["anchor"], {"positive": set(), "negative": set()})
        group["positive"].add(row["positive"])
        group["negative"].add(row["negative"])

    queries = {}
    corpus = {}
    relevant_docs = {}
    doc_id_by_text = {}

    def add_passage(text):
        text = "passage: " + text
        if text not in doc_id_by_text:
            doc_id = f"d{len(doc_id_by_text)}"
            doc_id_by_text[text] = doc_id
            corpus[doc_id] = text
        return doc_id_by_text[text]

    skipped_queries = 0
    for anchor in sorted(grouped):
        positives = sorted(grouped[anchor]["positive"])
        negatives = sorted(
            text
            for text in grouped[anchor]["negative"]
            if text not in training_texts and text not in positives
        )
        if not positives or not negatives:
            skipped_queries += 1
            continue

        qid = f"q{len(queries)}"
        queries[qid] = "query: " + anchor
        relevant_docs[qid] = {add_passage(text) for text in positives}

        query_rng = random.Random(f"{SEED}:{anchor}")
        selected_negatives = query_rng.sample(
            negatives, k=min(negatives_per_query, len(negatives))
        )
        for text in selected_negatives:
            add_passage(text)

    assert queries, "Aucune requête held-out évaluable."
    assert not (set(corpus.values()) & {"passage: " + x for x in training_texts})
    print({
        "heldout_queries_evaluated": len(queries),
        "heldout_queries_skipped": skipped_queries,
        "deduplicated_corpus_size": len(corpus),
    })

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        score_functions={"cosine": cos_sim},
        show_progress_bar=True,
    )

retrieval_evaluator = build_ir_evaluator(
    eval_ds,
    training_texts=all_training_texts,
    negatives_per_query=EVAL_NEGATIVES_PER_QUERY,
)

# Validation STS sans OpusParcus et sans aucun texte vu au train.
sts_validation = fr_sts_splits["validation"].filter(
    lambda row: row["dataset"] != "opusparcus"
    and row["sentence1"] not in all_training_texts
    and row["sentence2"] not in all_training_texts,
    desc="STS validation anti-leak",
).shuffle(seed=SEED)
sts_validation = sts_validation.select(range(min(STS_EVAL_SIZE, len(sts_validation))))
assert len(sts_validation) > 0

sts_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=["query: " + text for text in sts_validation["sentence1"]],
    sentences2=["query: " + text for text in sts_validation["sentence2"]],
    scores=sts_validation["score"],
    main_similarity=SimilarityFunction.COSINE,
    name="fr-sts-heldout",
)

# Le meilleur adapter doit progresser sans sacrifier la similarité sémantique.
evaluator = SequentialEvaluator(
    [retrieval_evaluator, sts_evaluator],
    main_score_function=lambda scores: float(np.mean(scores)),
)
print({"sts_heldout_rows": len(sts_validation)})
print("Evaluateur combiné anti-leak OK")

{'heldout_queries_evaluated': 200, 'heldout_queries_skipped': 0, 'deduplicated_corpus_size': 2006}


STS validation anti-leak:   0%|          | 0/3526 [00:00<?, ? examples/s]

{'sts_heldout_rows': 1173}
Evaluateur combiné anti-leak OK


In [9]:
# Cellule 8 — baseline à figer avant entraînement

baseline_metrics = evaluator(model)

print(baseline_metrics)

baseline_score = baseline_metrics["sequential_score"]
baseline_retrieval_score = baseline_metrics["miracl-fr-heldout_cosine_ndcg@10"]

print({
    "baseline_combined_score": baseline_score,
    "baseline_retrieval_ndcg@10": baseline_retrieval_score,
})

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:06<00:00,  6.55s/it]


{'miracl-fr-heldout_cosine_accuracy@1': 0.815, 'miracl-fr-heldout_cosine_accuracy@3': 0.955, 'miracl-fr-heldout_cosine_accuracy@5': 0.985, 'miracl-fr-heldout_cosine_accuracy@10': 1.0, 'miracl-fr-heldout_cosine_precision@1': 0.815, 'miracl-fr-heldout_cosine_precision@3': 0.5283333333333333, 'miracl-fr-heldout_cosine_precision@5': 0.37900000000000006, 'miracl-fr-heldout_cosine_precision@10': 0.20450000000000002, 'miracl-fr-heldout_cosine_recall@1': 0.502436507936508, 'miracl-fr-heldout_cosine_recall@3': 0.8328095238095239, 'miracl-fr-heldout_cosine_recall@5': 0.942015873015873, 'miracl-fr-heldout_cosine_recall@10': 0.9963333333333333, 'miracl-fr-heldout_cosine_ndcg@10': 0.896914037702482, 'miracl-fr-heldout_cosine_mrr@10': 0.8903333333333334, 'miracl-fr-heldout_cosine_map@100': 0.8443801186467853, 'fr-sts-heldout_pearson_cosine': 0.8141589272761572, 'fr-sts-heldout_spearman_cosine': 0.8129129420918769, 'sequential_score': 0.8549134898971795}
{'baseline_combined_score': 0.8549134898971795

In [10]:
# Cellule 9 — test hard negative avant entraînement

row = train_ds[0]

q = "query: " + row["anchor"]
p = "passage: " + row["positive"]
n = "passage: " + row["negative"]

emb = model.encode([q, p, n], normalize_embeddings=True)

positive_score = float(emb[0] @ emb[1])
negative_score = float(emb[0] @ emb[2])

print({
    "positive_score": positive_score,
    "negative_score": negative_score,
    "margin": positive_score - negative_score,
})

assert positive_score > negative_score, "La baseline ne classe pas ce triplet correctement"

{'positive_score': 0.8542371988296509, 'negative_score': 0.8373702764511108, 'margin': 0.01686692237854004}


In [11]:
# Cellule 10 — création du vrai PeftModel AVANT le trainer et l'optimiseur
# get_peft_model est utilisé explicitement pour fiabiliser save/load avec la pile installée.

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # Toutes les projections des 12 blocs, mais pas le pooler XLM-R inutilisé.
    target_modules=(
        r".*encoder\.layer\.\d+\."
        r"(attention\.self\.(query|key|value)|attention\.output\.dense|"
        r"intermediate\.dense|output\.dense)"
    ),
    bias="none",
)

model[0].model = get_peft_model(model[0].auto_model, lora_config)
assert isinstance(model[0].auto_model, PeftModel)
print("Backbone PEFT:", type(model[0].auto_model).__name__)

Backbone PEFT: PeftModelForFeatureExtraction


In [12]:
# Cellule 11 — garde-fous AVANT entraînement

backbone = model[0].auto_model
lora_all = [(name, p) for name, p in backbone.named_parameters() if "lora_" in name.lower()]
lora_trainable = [(name, p) for name, p in lora_all if p.requires_grad]
non_lora_trainable = [
    name for name, p in backbone.named_parameters()
    if p.requires_grad and "lora_" not in name.lower()
]
lora_b_before = {
    name: p.detach().float().cpu().clone()
    for name, p in lora_all if "lora_b" in name.lower()
}

trainable = sum(p.numel() for _, p in lora_trainable)
total = sum(p.numel() for p in model.parameters())
initial_b_max = max(t.abs().max().item() for t in lora_b_before.values())

print("Tenseurs LoRA:", len(lora_all))
print("Paramètres LoRA entraînables:", f"{trainable:,}")
print("Part entraînable:", f"{100 * trainable / total:.3f}%")
print("max abs LoRA_B initial:", initial_b_max)

assert lora_all and lora_trainable, "LoRA absent ou gelé."
assert not non_lora_trainable, f"Paramètres non-LoRA entraînables: {non_lora_trainable[:5]}"
assert initial_b_max == 0.0, "LoRA_B devrait être nul à l'initialisation."

Tenseurs LoRA: 144
Paramètres LoRA entraînables: 2,654,208
Part entraînable: 0.946%
max abs LoRA_B initial: 0.0


In [13]:
# Cellule 11 — pertes adaptées à chaque type de supervision

retrieval_loss = CachedMultipleNegativesRankingLoss(
    model=model,
    mini_batch_size=8,
)
similarity_loss = CoSENTLoss(model=model)

loss = {
    "miracl": retrieval_loss,
    "fr_sts": similarity_loss,
    "fr_paraphrase": retrieval_loss,
    "fr_nli": similarity_loss,
    "multilingual_replay": retrieval_loss,
}

assert set(loss) == set(train_datasets)
print({name: type(dataset_loss).__name__ for name, dataset_loss in loss.items()})

{'miracl': 'CachedMultipleNegativesRankingLoss', 'fr_sts': 'CoSENTLoss', 'fr_paraphrase': 'CachedMultipleNegativesRankingLoss', 'fr_nli': 'CoSENTLoss', 'multilingual_replay': 'CachedMultipleNegativesRankingLoss'}


In [14]:
# Cellule 12 — arguments d'entraînement

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_RATIO,

    fp16=(device == "cuda"),
    bf16=False,

    batch_sampler=BatchSamplers.NO_DUPLICATES,
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,

    eval_strategy="steps",
    eval_steps=250,

    # Ne pas utiliser les checkpoints ST automatiques pour ce PEFT imbriqué.
    # Avec les versions installées, leur rechargement remet LoRA_B à zéro.
    save_strategy="no",

    logging_steps=25,

    load_best_model_at_end=False,

    report_to="none",

    # Prompts E5 par dataset: retrieval asymétrique, autres tâches symétriques.
    prompts={
        "miracl": {
            "anchor": "query: ",
            "positive": "passage: ",
            "negative": "passage: ",
        },
        "fr_sts": {"sentence1": "query: ", "sentence2": "query: "},
        "fr_paraphrase": {"anchor": "query: ", "positive": "query: "},
        "fr_nli": {"sentence1": "query: ", "sentence2": "query: "},
        "multilingual_replay": {"anchor": "query: ", "positive": "query: "},
    },

    dataloader_num_workers=0,
    seed=SEED,
)

print("Arguments OK")

Arguments OK


In [15]:
class LoraTrainingGuardCallback(TrainerCallback):
    """Vérifie LoRA_B et conserve le meilleur adapter sur le holdout propre."""

    def __init__(self, best_adapter_dir, metric_name):
        self.gradient_checked = False
        self.weight_checked = False
        self.best_adapter_dir = best_adapter_dir
        self.metric_name = metric_name
        self.best_score = float("-inf")
        self.best_step = None
        self.best_lora_state = None

    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        # Ce hook voit les gradients avant optimizer.step() et model.zero_grad().
        if not self.gradient_checked:
            lora_b_gradients = [
                p.grad.detach().float()
                for name, p in model[0].auto_model.named_parameters()
                if "lora_b" in name.lower() and p.grad is not None
            ]
            grad_max = max((g.abs().max().item() for g in lora_b_gradients), default=0.0)
            print(f"[LoRA gradient check] max abs grad LoRA_B = {grad_max:.3e}")
            if not np.isfinite(grad_max) or grad_max <= 0.0:
                raise RuntimeError(
                    f"Gradients LoRA_B invalides avant optimizer.step(): {grad_max}"
                )
            self.gradient_checked = True
        return control

    def on_optimizer_step(self, args, state, control, model=None, optimizer=None, **kwargs):
        # Avec warmup, le LR du premier step vaut 0: attendre le premier LR positif.
        learning_rate = max(group["lr"] for group in optimizer.param_groups)
        if not self.weight_checked and learning_rate > 0.0:
            lora_b = [
                p.detach().float()
                for name, p in model[0].auto_model.named_parameters()
                if "lora_b" in name.lower()
            ]
            max_abs = max(p.abs().max().item() for p in lora_b)
            print(
                f"[LoRA weight check step {state.global_step + 1}] "
                f"lr={learning_rate:.3e}, max abs LoRA_B={max_abs:.3e}"
            )
            if not np.isfinite(max_abs) or max_abs <= 0.0:
                raise RuntimeError(
                    f"LoRA_B invalide malgré un optimizer step à LR positif: {max_abs}"
                )
            self.weight_checked = True
        return control

    def consider(self, model, score, step):
        if score <= self.best_score:
            return False
        backbone = model[0].auto_model
        self.best_score = float(score)
        self.best_step = step
        self.best_lora_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in backbone.state_dict().items()
            if "lora_" in name.lower()
        }
        backbone.save_pretrained(self.best_adapter_dir, safe_serialization=True)
        print(f"[best adapter] step={step} combined_score={score:.6f}")
        return True

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = (metrics or {}).get(self.metric_name)
        if score is not None:
            self.consider(model, score, state.global_step)
        return control

    def restore_best(self, model):
        assert self.best_lora_state is not None, "Aucun adapter validé."
        model[0].auto_model.load_state_dict(self.best_lora_state, strict=False)
        print(f"Adapter restauré: step={self.best_step}, score={self.best_score:.6f}")


BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best-adapter-heldout")
HELDOUT_METRIC = "eval_sequential_score"
lora_training_guard = LoraTrainingGuardCallback(BEST_ADAPTER_DIR, HELDOUT_METRIC)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_datasets,
    loss=loss,
    evaluator=evaluator,
    callbacks=[lora_training_guard],
)

# Construction explicite de l'optimiseur, puis preuve que tous les LoRA trainables y sont.
trainer.create_optimizer()
optimizer_param_ids = {
    id(p) for group in trainer.optimizer.param_groups for p in group["params"]
}
missing_from_optimizer = [name for name, p in lora_trainable if id(p) not in optimizer_param_ids]
assert not missing_from_optimizer, f"LoRA absent de l'optimiseur: {missing_from_optimizer[:5]}"
print("Optimiseur OK: tous les paramètres LoRA entraînables sont présents.")

trainer.train()

lora_b_after = {
    name: p.detach().float().cpu()
    for name, p in model[0].auto_model.named_parameters()
    if "lora_b" in name.lower()
}
max_b_after = max(t.abs().max().item() for t in lora_b_after.values())
max_b_delta = max(
    (tensor - lora_b_before[name]).abs().max().item()
    for name, tensor in lora_b_after.items()
)

assert lora_training_guard.gradient_checked
assert lora_training_guard.weight_checked
assert np.isfinite(max_b_after) and np.isfinite(max_b_delta)
assert max_b_after > 0.0 and max_b_delta > 0.0
print({"max_abs_LoRA_B": max_b_after, "max_delta_LoRA_B": max_b_delta})
print("Training terminé: les poids LoRA ont réellement changé.")

Computing widget examples:   0%|          | 0/3 [00:00<?, ?example/s]

Optimiseur OK: tous les paramètres LoRA entraînables sont présents.
[LoRA gradient check] max abs grad LoRA_B = 2.529e-01


Step,Training Loss,Validation Loss,Miracl-fr-heldout Cosine Accuracy@1,Miracl-fr-heldout Cosine Accuracy@3,Miracl-fr-heldout Cosine Accuracy@5,Miracl-fr-heldout Cosine Accuracy@10,Miracl-fr-heldout Cosine Precision@1,Miracl-fr-heldout Cosine Precision@3,Miracl-fr-heldout Cosine Precision@5,Miracl-fr-heldout Cosine Precision@10,Miracl-fr-heldout Cosine Recall@1,Miracl-fr-heldout Cosine Recall@3,Miracl-fr-heldout Cosine Recall@5,Miracl-fr-heldout Cosine Recall@10,Miracl-fr-heldout Cosine Ndcg@10,Miracl-fr-heldout Cosine Mrr@10,Miracl-fr-heldout Cosine Map@100,Fr-sts-heldout Pearson Cosine,Fr-sts-heldout Spearman Cosine,Sequential Score
250,1.741070,No log,0.835000,0.960000,0.975000,1.000000,0.835000,0.526667,0.376000,0.204500,0.509520,0.831393,0.931099,0.997444,0.898832,0.901012,0.843611,0.802954,0.807644,0.853238
500,2.132802,No log,0.845000,0.965000,0.980000,1.000000,0.845000,0.535000,0.376000,0.204500,0.517853,0.845310,0.935433,0.995500,0.904024,0.904958,0.853440,0.821392,0.824327,0.864175
750,2.057176,No log,0.830000,0.960000,0.980000,0.995000,0.830000,0.533333,0.375000,0.203500,0.512020,0.841143,0.936099,0.991333,0.898591,0.895458,0.848970,0.825109,0.834096,0.866344
1000,2.700197,No log,0.830000,0.945000,0.975000,0.990000,0.830000,0.526667,0.375000,0.202500,0.515353,0.825143,0.934016,0.986333,0.896589,0.892083,0.849726,0.832645,0.835669,0.866129
1250,1.262781,No log,0.820000,0.950000,0.975000,0.995000,0.820000,0.526667,0.375000,0.203500,0.508687,0.827810,0.933349,0.991333,0.895389,0.888583,0.845318,0.822864,0.829917,0.862653
1436,2.353064,No log,0.825000,0.950000,0.975000,0.990000,0.825000,0.526667,0.375000,0.202500,0.510353,0.827810,0.933349,0.986333,0.893718,0.890625,0.844685,0.821374,0.828869,0.861293


[LoRA weight check step 2] lr=3.472e-07, max abs LoRA_B=3.477e-07


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:12<00:00, 12.93s/it]


{'max_abs_LoRA_B': 0.014681515283882618, 'max_delta_LoRA_B': 0.014681515283882618}
Training terminé: les poids LoRA ont réellement changé.


In [16]:
# Cellule 15 — sélection et verdict sur le holdout sans fuite

last_metrics = evaluator(model)
last_score = last_metrics["sequential_score"]
lora_training_guard.consider(model, last_score, trainer.state.global_step)
lora_training_guard.restore_best(model)

# Recalcul après restauration: le modèle ensuite sauvegardé est bien le meilleur.
selected_metrics = evaluator(model)
final_score = selected_metrics["sequential_score"]
assert abs(final_score - lora_training_guard.best_score) < 1e-8
delta = final_score - baseline_score

print({
    "baseline_combined_score": round(baseline_score, 4),
    "last_step_combined_score": round(last_score, 4),
    "selected_combined_score": round(final_score, 4),
    "selected_retrieval_ndcg@10": round(
        selected_metrics["miracl-fr-heldout_cosine_ndcg@10"], 4
    ),
    "selected_step": lora_training_guard.best_step,
    "delta": round(delta, 4),
})

if final_score > baseline_score:
    print("WIN combiné held-out: candidat valable pour MTEB français")
else:
    print("REGRESSION held-out: ne pas lancer MTEB complet")

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:10<00:00, 10.76s/it]


Adapter restauré: step=750, score=0.866344


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:19<00:00, 19.73s/it]


{'baseline_combined_score': 0.8549, 'last_step_combined_score': 0.8613, 'selected_combined_score': 0.8663, 'selected_retrieval_ndcg@10': 0.8986, 'selected_step': 750, 'delta': 0.0114}
WIN combiné held-out: candidat valable pour MTEB français


In [17]:
# Cellule 16 — test qualitatif final

tests = [
    {
        "query": "Qui était la princesse Diana ?",
        "positive": "Diana Spencer était une membre de la famille royale britannique.",
        "negative": "La photosynthèse permet aux plantes de produire de l'énergie.",
    },
    {
        "query": "Quelle est la capitale de la France ?",
        "positive": "Paris est la capitale de la France.",
        "negative": "Le Japon est un archipel situé en Asie de l'Est.",
    },
]

for item in tests:
    vectors = model.encode(
        [
            "query: " + item["query"],
            "passage: " + item["positive"],
            "passage: " + item["negative"],
        ],
        normalize_embeddings=True,
    )

    pos = float(vectors[0] @ vectors[1])
    neg = float(vectors[0] @ vectors[2])

    print({
        "query": item["query"],
        "positive": round(pos, 4),
        "negative": round(neg, 4),
        "margin": round(pos - neg, 4),
    })

    assert pos > neg

{'query': 'Qui était la princesse Diana ?', 'positive': 0.515, 'negative': -0.0876, 'margin': 0.6026}
{'query': 'Quelle est la capitale de la France ?', 'positive': 0.6934, 'negative': 0.116, 'margin': 0.5774}


In [4]:

import shutil

ADAPTER_DIR = "./e5-fr-miracl-lora-adapter"
FINAL_DIR = "./e5-fr-miracl-lora-merged"
probe_texts = [
    "query: Quelle est la capitale de la France ?",
    "passage: Paris est la capitale de la France.",
]

In [ ]:
# Cellule 17 — export robuste depuis le meilleur adapter sauvegardé
# On ne dépend pas du type de wrapper laissé en mémoire par Trainer/Accelerate.



assert os.path.isfile(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json"))
assert os.path.isfile(os.path.join(BEST_ADAPTER_DIR, "adapter_model.safetensors"))

# Libère les états optimizer/gradients GPU devenus inutiles.
model.to("cpu")
del trainer, loss, retrieval_loss, similarity_loss, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Recharge explicitement le meilleur adapter sur un backbone E5 propre, en CPU.
export_model = SentenceTransformer(MODEL_NAME, device="cpu")
export_model.max_seq_length = MAX_SEQ_LENGTH
export_model[0].model = PeftModel.from_pretrained(
    export_model[0].auto_model,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)
peft_backbone = export_model[0].auto_model
lora_b_export = [
    parameter.detach().float()
    for name, parameter in peft_backbone.named_parameters()
    if "lora_b" in name.lower()
]
max_lora_b_export = max((p.abs().max().item() for p in lora_b_export), default=0.0)
assert np.isfinite(max_lora_b_export) and max_lora_b_export > 0.0

reference_embeddings = export_model.encode(probe_texts, normalize_embeddings=True)

# Supprime d'éventuels fichiers résiduels d'anciens exports cassés.
for output_path in (ADAPTER_DIR, FINAL_DIR):
    if os.path.isdir(output_path):
        shutil.rmtree(output_path)

peft_backbone.save_pretrained(ADAPTER_DIR, safe_serialization=True)
export_model[0].model = peft_backbone.merge_and_unload(safe_merge=True)
export_model.save_pretrained(FINAL_DIR)

reloaded_model = SentenceTransformer(FINAL_DIR, device="cpu")
reloaded_embeddings = reloaded_model.encode(probe_texts, normalize_embeddings=True)
reload_max_diff = float(np.max(np.abs(reference_embeddings - reloaded_embeddings)))
assert np.allclose(reference_embeddings, reloaded_embeddings, rtol=1e-4, atol=1e-5)

# Le modèle global redevient le modèle fusionné propre, en CPU.
model = export_model
del reloaded_model
gc.collect()

print({
    "best_step": lora_training_guard.best_step,
    "best_combined_score": lora_training_guard.best_score,
    "max_abs_LoRA_B_export": max_lora_b_export,
    "reload_max_diff": reload_max_diff,
})
print("Adapter PEFT:", ADAPTER_DIR)
print("Modèle fusionné MTEB:", FINAL_DIR)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'best_step': 750, 'best_combined_score': 0.866343571246924, 'max_abs_LoRA_B_export': 0.012302129529416561, 'reload_max_diff': 2.1373853087425232e-07}
Adapter PEFT: ./e5-fr-miracl-lora-adapter
Modèle fusionné MTEB: ./e5-fr-miracl-lora-merged


## 10. Rechargement et vérification

In [ ]:
import ctypes
import gc
import os
import torch
from sentence_transformers import SentenceTransformer

# Le training/export laisse plusieurs alias vers des backbones E5 en mémoire.
# On les supprime avant MTEB pour ne conserver qu'un seul modèle sur le GPU.
for _name in (
    "model", "base_model", "st_model", "export_model",
    "reloaded_model", "adapter_model", "benchmark", "task", "results",
    "peft_backbone", "backbone", "lora_b", "lora_b_export",
    "baseline_model", "finetuned_model", "adapter_embeddings",
    "merged_embeddings", "reference_embeddings", "reloaded_embeddings",
    "baseline_probe", "finetuned_probe", "test_q", "test_d",
    "ds", "fr_sts_splits", "fr_nli_raw", "replay_raw",
    "train_ds", "eval_ds", "sts_source", "sts_train",
    "paraphrase_source", "paraphrase_train", "nli_source", "nli_parts",
    "nli_train", "replay_parts", "replay_train", "train_datasets",
    "all_training_texts", "heldout_texts", "eval_positive_texts",
    "retrieval_evaluator", "sts_validation", "sts_evaluator", "evaluator",
    "lora_b_before", "lora_b_after", "q_emb", "p_emb", "emb",
    "lora_training_guard",
):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if os.name == "nt":
    ctypes.windll.psapi.EmptyWorkingSet(ctypes.windll.kernel32.GetCurrentProcess())

# Valeurs autonomes: cette section peut être lancée après un restart kernel,
# sans réexécuter le téléchargement des datasets ni le training.
MODEL_NAME = globals().get("MODEL_NAME", "intfloat/multilingual-e5-base")
ADAPTER_DIR = globals().get("ADAPTER_DIR", "./e5-fr-miracl-lora-adapter")
FINAL_DIR = globals().get("FINAL_DIR", "./e5-fr-miracl-lora-merged")
MAX_SEQ_LENGTH = globals().get("MAX_SEQ_LENGTH", 512)
device = "cuda" if torch.cuda.is_available() else "cpu"

BASELINE_DIR_OR_NAME = MODEL_NAME
FINETUNED_DIR = FINAL_DIR

# MTEB n'a besoin que du modèle fusionné. Le baseline et l'adapter ne sont pas
# rechargés ici: cela évite 1 à 2 Go de RAM temporaire que Python rend mal à Windows.
# FP16 économise ~0.5 Go de VRAM mais modifie sensiblement certains clusterings.
USE_FP16_EVAL = False
EVAL_TORCH_DTYPE = (
    torch.float16 if USE_FP16_EVAL and torch.cuda.is_available() else torch.float32
)
LOAD_MODEL_IN_NOTEBOOK = False
if LOAD_MODEL_IN_NOTEBOOK:
    finetuned_model = SentenceTransformer(
        FINETUNED_DIR,
        device=device,
        model_kwargs={"torch_dtype": EVAL_TORCH_DTYPE},
    )
else:
    finetuned_model = None
    print("Mode SAFE: aucun modèle conservé dans le kernel Jupyter.")


In [ ]:
if finetuned_model is not None:
    merged_lora_params = [
        name for name, _ in finetuned_model[0].auto_model.named_parameters()
        if "lora_" in name.lower()
    ]
    assert not merged_lora_params
    print("Modèle fusionné sans couche LoRA résiduelle.")
else:
    print("Vérification locale ignorée en mode SAFE.")

In [ ]:
# Diagnostic lourd facultatif. Laisser False pendant MTEB.
RUN_RELOAD_DIAGNOSTICS = False
if RUN_RELOAD_DIAGNOSTICS:
    from peft import PeftModel
    adapter_model = SentenceTransformer(MODEL_NAME, device="cpu")
    adapter_model.max_seq_length = MAX_SEQ_LENGTH
    adapter_model[0].model = PeftModel.from_pretrained(
        adapter_model[0].auto_model, ADAPTER_DIR, is_trainable=False
    )
    adapter_model.eval()
else:
    print("Diagnostic adapter ignoré pour économiser la RAM.")

In [ ]:
if RUN_RELOAD_DIAGNOSTICS:
    backbone = adapter_model[0].auto_model
    lora_b = [
        (name, p.detach().float())
        for name, p in backbone.named_parameters()
        if "lora_b" in name.lower()
    ]
    max_lora_b = max(p.abs().max().item() for _, p in lora_b)
    assert lora_b and max_lora_b > 0.0
    adapter_embeddings = adapter_model.encode(probe_texts, normalize_embeddings=True)
    merged_embeddings = finetuned_model.encode(probe_texts, normalize_embeddings=True)
    assert np.allclose(adapter_embeddings, merged_embeddings, rtol=1e-3, atol=2e-4)
    print("Adapter PEFT et modèle FP16 fusionné: embeddings cohérents.")
    del adapter_model, backbone, lora_b, adapter_embeddings, merged_embeddings
    gc.collect()
else:
    print("Validation adapter déjà effectuée pendant l'export.")

In [ ]:
if finetuned_model is not None:
    print("Fine-tuned dim:", finetuned_model.get_embedding_dimension())
    print("dtype évaluation:", next(finetuned_model.parameters()).dtype)
else:
    print("Le worker isolé chargera et libérera le modèle tâche par tâche.")

In [ ]:
# L'évaluation SAFE est autonome dans la cellule suivante.
# Aucun import MTEB/PyTorch lourd n'est nécessaire dans le kernel Jupyter.
print("Préparation du superviseur MTEB SAFE.")

In [ ]:
import hashlib

def short_file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()[:12]

MODEL_REVISION = "lora-" + short_file_sha256(
    os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
)
MODEL_LABEL = "local/multilingual-e5-base-fr-multitask-lora"
RUN_NAME = f"e5_fr_multitask_{MODEL_REVISION}"
print({"model_label": MODEL_LABEL, "revision": MODEL_REVISION, "run": RUN_NAME})

In [ ]:
# Pour mesurer la baseline dans un second run, utiliser à la place :
# MODEL_LABEL = MODEL_NAME
# MODEL_REVISION = "upstream-baseline"
# RUN_NAME = "e5_baseline_upstream"
# base_model = baseline_model

In [1]:
# Evaluation MTEB SAFE: une tache par processus jetable.
# Cette cellule est autonome apres un restart kernel et reprend les JSON existants.
import ctypes
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import psutil

MODEL_NAME = "intfloat/multilingual-e5-base"
MODEL_LABEL = "local/multilingual-e5-base-fr-multitask-lora"
BASE_DIR = Path.cwd()
if not (BASE_DIR / "mteb_safe_worker.py").is_file():
    candidate = Path(r"F:\codeAI\llm\emebedding")
    if (candidate / "mteb_safe_worker.py").is_file():
        BASE_DIR = candidate

FINETUNED_DIR = (BASE_DIR / "e5-fr-miracl-lora-merged").resolve()
ADAPTER_FILE = (BASE_DIR / "e5-fr-miracl-lora-adapter" / "adapter_model.safetensors").resolve()
WORKER_PATH = (BASE_DIR / "mteb_safe_worker.py").resolve()
PYTHON_EXECUTABLE = BASE_DIR.parent / ".venv" / "Scripts" / "python.exe"
if not PYTHON_EXECUTABLE.is_file():
    PYTHON_EXECUTABLE = Path(sys.executable)

assert FINETUNED_DIR.is_dir(), FINETUNED_DIR
assert ADAPTER_FILE.is_file(), ADAPTER_FILE
assert WORKER_PATH.is_file(), WORKER_PATH

def short_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()[:12]

MODEL_REVISION = "lora-" + short_sha256(ADAPTER_FILE)
RUN_NAME = f"e5_fr_multitask_{MODEL_REVISION}"
BENCHMARK_LABEL = "MTEB(fra)-safe v2 CPU-only"
OUTPUT_FOLDER = BASE_DIR / "mteb_fra_results" / RUN_NAME
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER = OUTPUT_FOLDER / "logs"
LOG_FOLDER.mkdir(exist_ok=True)

# Les trois taches legacy responsables des crashes chargent de tres gros datasets.
# Leurs versions v2 officielles MTEB utilisent un echantillon borne a 2 048 textes.
SAFE_TASKS = [
    "AmazonReviewsClassification",
    "MasakhaNEWSClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "MTOPIntentClassification",
    "AlloProfClusteringP2P",
    "AlloProfClusteringS2S",
    "HALClusteringS2S.v2",
    "MasakhaNEWSClusteringP2P",
    "MasakhaNEWSClusteringS2S",
    "MLSUMClusteringP2P.v2",
    "MLSUMClusteringS2S.v2",
    "PawsXPairClassification",
    "AlloprofReranking",
    "SyntecReranking",
    "AlloprofRetrieval",
    "BSARDRetrieval",
    "MintakaRetrieval",
    "SyntecRetrieval",
    "XPQARetrieval",
    "SICKFr",
    "STSBenchmarkMultilingualSTS",
    "SummEvalFr",
    "STS22",
]

# Reglages conservateurs pour RTX 3070 / 16 Go RAM.
CPU_ONLY = True
CPU_THREADS = 4
SAFE_BATCH_SIZE = 16
SAFE_CHUNK_SIZE = 256
WORKER_COOLDOWN_SECONDS = 0.0
MIN_FREE_RAM_GB = 0.75
MAX_WORKER_RSS_GB = 5.0
MAX_GPU_TEMP_C = 78
GPU_TASK_TIMEOUT_MIN = 10
CPU_TASK_TIMEOUT_MIN = 360
POLL_SECONDS = 3
RETRY_ON_CPU = False


def result_path(task_name):
    return OUTPUT_FOLDER / f"{task_name}.json"


def error_path(task_name):
    return OUTPUT_FOLDER / f"{task_name}_ERROR.json"


def valid_result(task_name):
    path = result_path(task_name)
    if not path.is_file():
        return False
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        return (
            data.get("run_name") == RUN_NAME
            and data.get("model_label") == MODEL_LABEL
            and data.get("model_revision") == MODEL_REVISION
            and isinstance(data.get("score"), (int, float))
        )
    except (OSError, json.JSONDecodeError):
        return False


def process_tree_rss_gb(pid):
    try:
        root = psutil.Process(pid)
        processes = [root] + root.children(recursive=True)
        return sum(p.memory_info().rss for p in processes if p.is_running()) / 1024**3
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return 0.0


def stop_process_tree(pid):
    try:
        root = psutil.Process(pid)
        processes = root.children(recursive=True) + [root]
    except psutil.NoSuchProcess:
        return
    for process in processes:
        try:
            process.terminate()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    _, alive = psutil.wait_procs(processes, timeout=5)
    for process in alive:
        try:
            process.kill()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass


def gpu_temperature():
    try:
        output = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=temperature.gpu",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            timeout=3,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
        )
        return int(output.strip().splitlines()[0])
    except Exception:
        return None


def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temporary, path)


def run_worker(task_name, device):
    log_path = LOG_FOLDER / f"{task_name}_{device}.log"
    command = [
        str(PYTHON_EXECUTABLE),
        str(WORKER_PATH),
        "--task", task_name,
        "--model-dir", str(FINETUNED_DIR),
        "--model-label", MODEL_LABEL,
        "--model-revision", MODEL_REVISION,
        "--adapted-from", MODEL_NAME,
        "--result-path", str(result_path(task_name)),
        "--run-name", RUN_NAME,
        "--benchmark-label", BENCHMARK_LABEL,
        "--device", device,
        "--batch-size", str(SAFE_BATCH_SIZE),
        "--chunk-size", str(SAFE_CHUNK_SIZE),
        "--cooldown-seconds", str(WORKER_COOLDOWN_SECONDS),
        "--min-free-ram-gb", str(MIN_FREE_RAM_GB),
        "--cpu-threads", str(CPU_THREADS),
    ]
    environment = os.environ.copy()
    environment.update({
        "OMP_NUM_THREADS": str(CPU_THREADS),
        "MKL_NUM_THREADS": str(CPU_THREADS),
        "OPENBLAS_NUM_THREADS": str(CPU_THREADS),
        "TOKENIZERS_PARALLELISM": "false",
    })
    if device == "cpu":
        environment["CUDA_VISIBLE_DEVICES"] = ""
    creationflags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
    timeout_seconds = 60 * (
        GPU_TASK_TIMEOUT_MIN if device == "cuda" else CPU_TASK_TIMEOUT_MIN
    )

    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=str(BASE_DIR),
            env=environment,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            creationflags=creationflags,
        )
        started = time.monotonic()
        last_report = started
        reason = None

        try:
            while process.poll() is None:
                time.sleep(POLL_SECONDS)
                elapsed = time.monotonic() - started
                free_ram_gb = psutil.virtual_memory().available / 1024**3
                worker_rss_gb = process_tree_rss_gb(process.pid)
                temperature = gpu_temperature() if device == "cuda" else None
    
                if free_ram_gb < MIN_FREE_RAM_GB:
                    reason = f"RAM systeme basse ({free_ram_gb:.2f} Go)"
                elif worker_rss_gb > MAX_WORKER_RSS_GB:
                    reason = f"worker trop volumineux ({worker_rss_gb:.2f} Go)"
                elif temperature is not None and temperature >= MAX_GPU_TEMP_C:
                    reason = f"GPU trop chaud ({temperature} C)"
                elif elapsed > timeout_seconds:
                    reason = f"timeout ({elapsed / 60:.1f} min)"
    
                if reason:
                    stop_process_tree(process.pid)
                    break
    
                if time.monotonic() - last_report >= 30:
                    temp_text = f", GPU {temperature} C" if temperature is not None else ""
                    print(
                        f"    {elapsed / 60:.1f} min, worker {worker_rss_gb:.1f} Go, "
                        f"RAM libre {free_ram_gb:.1f} Go{temp_text}",
                        flush=True,
                    )
                    last_report = time.monotonic()
    
        except KeyboardInterrupt:
            stop_process_tree(process.pid)
            raise

        return_code = process.poll()

    if reason:
        return False, reason, log_path
    if return_code != 0:
        return False, f"worker exit={return_code}", log_path
    if not valid_result(task_name):
        return False, "resultat absent ou invalide", log_path
    return True, "ok", log_path


# Le kernel Jupyter ne doit garder ni modele ni dataset.
for variable in (
    "model", "base_model", "finetuned_model", "baseline_model", "adapter_model",
    "st_model", "benchmark", "task", "results", "train_ds", "eval_ds",
    "train_datasets", "fr_sts_splits", "fr_nli_raw", "replay_raw",
):
    globals().pop(variable, None)
import gc
gc.collect()
if os.name == "nt":
    ctypes.windll.psapi.EmptyWorkingSet(ctypes.windll.kernel32.GetCurrentProcess())

completed = [task for task in SAFE_TASKS if valid_result(task)]
pending = [task for task in SAFE_TASKS if not valid_result(task)]
print({
    "benchmark": BENCHMARK_LABEL,
    "model_revision": MODEL_REVISION,
    "completed": len(completed),
    "pending": len(pending),
    "worker": str(WORKER_PATH),
})

for index, task_name in enumerate(pending, start=1):
    primary_device = "cpu" if CPU_ONLY else "cuda"
    print(
        f"[{index}/{len(pending)}] {task_name} - worker {primary_device.upper()} isole",
        flush=True,
    )
    success, reason, log_path = run_worker(task_name, primary_device)

    if not success and RETRY_ON_CPU and primary_device != "cpu":
        print(f"    GPU interrompu: {reason}; nouvelle tentative CPU", flush=True)
        time.sleep(5)
        success, reason, log_path = run_worker(task_name, "cpu")

    if success:
        data = json.loads(result_path(task_name).read_text(encoding="utf-8"))
        print(f"    OK {data['score']:.4f} ({data['eval_device']})", flush=True)
        error_path(task_name).unlink(missing_ok=True)
    else:
        payload = {
            "run_name": RUN_NAME,
            "task": task_name,
            "status": "failed_safely",
            "reason": reason,
            "log": str(log_path),
        }
        atomic_json(error_path(task_name), payload)
        print(f"    ECHEC SUR: {reason}; voir {log_path}", flush=True)

    current_completed = [task for task in SAFE_TASKS if valid_result(task)]
    atomic_json(
        OUTPUT_FOLDER / "run_status.json",
        {
            "run_name": RUN_NAME,
            "benchmark": BENCHMARK_LABEL,
            "completed": current_completed,
            "remaining": [task for task in SAFE_TASKS if task not in current_completed],
            "last_task": task_name,
        },
    )
    time.sleep(3)

all_scores = {}
for task_name in SAFE_TASKS:
    if valid_result(task_name):
        data = json.loads(result_path(task_name).read_text(encoding="utf-8"))
        all_scores[task_name] = float(data["score"])

print("\n" + "=" * 70)
if all_scores:
    df = pd.DataFrame(
        [{"task": task, "score": score} for task, score in all_scores.items()]
    ).sort_values("score", ascending=False).reset_index(drop=True)
    mean_score = float(df["score"].mean())
    df["score"] = df["score"].round(4)
    display(df)
    df.to_csv(OUTPUT_FOLDER / "summary.csv", index=False)
    atomic_json(
        OUTPUT_FOLDER / "summary.json",
        {
            "run_name": RUN_NAME,
            "model_label": MODEL_LABEL,
            "model_revision": MODEL_REVISION,
            "benchmark_label": BENCHMARK_LABEL,
            "official_mteb_fra_v1": False,
            "mean_score": mean_score,
            "completed_tasks": len(all_scores),
            "total_tasks": len(SAFE_TASKS),
        },
    )
    print(
        f"Score moyen {BENCHMARK_LABEL}: {mean_score:.4f} "
        f"({len(all_scores)}/{len(SAFE_TASKS)} taches)"
    )
    print(f"Resultats: {OUTPUT_FOLDER}")
else:
    print("Aucun score valide.")


{'benchmark': 'MTEB(fra)-safe v2 CPU-only', 'model_revision': 'lora-5701562d9417', 'completed': 14, 'pending': 11, 'worker': 'F:\\codeAI\\llm\\emebedding\\mteb_safe_worker.py'}
[1/11] AlloprofReranking - worker CPU isole
    0.5 min, worker 0.6 Go, RAM libre 3.8 Go
    1.0 min, worker 0.8 Go, RAM libre 3.6 Go
    1.5 min, worker 1.7 Go, RAM libre 2.9 Go
    2.0 min, worker 1.7 Go, RAM libre 2.9 Go
    2.5 min, worker 1.7 Go, RAM libre 2.9 Go
    3.0 min, worker 1.8 Go, RAM libre 2.9 Go
    3.5 min, worker 1.8 Go, RAM libre 2.9 Go
    4.0 min, worker 1.8 Go, RAM libre 2.8 Go
    4.5 min, worker 1.8 Go, RAM libre 2.8 Go
    5.0 min, worker 1.8 Go, RAM libre 2.8 Go
    5.5 min, worker 1.8 Go, RAM libre 2.7 Go
    6.0 min, worker 1.8 Go, RAM libre 3.3 Go
    6.5 min, worker 1.8 Go, RAM libre 3.3 Go
    7.0 min, worker 1.8 Go, RAM libre 3.3 Go
    7.5 min, worker 1.8 Go, RAM libre 3.3 Go
    8.0 min, worker 1.8 Go, RAM libre 3.3 Go
    8.5 min, worker 1.8 Go, RAM libre 3.3 Go
    9.0 min, w

,task,score
0,STSBenchmarkMultilingualSTS,0.8302
1,MTOPDomainClassification,0.8025
2,SICKFr,0.7744
3,MasakhaNEWSClassification,0.7659
4,MassiveScenarioClassification,0.6482
5,STS22,0.6187
6,MassiveIntentClassification,0.6098
7,PawsXPairClassification,0.5883
8,SyntecReranking,0.5442
9,MTOPIntentClassification,0.5244


Score moyen MTEB(fra)-safe v2 CPU-only: 0.4696 (25/25 taches)
Resultats: f:\codeAI\llm\emebedding\mteb_fra_results\e5_fr_multitask_lora-5701562d9417


In [2]:
import pandas as pd

df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "summary.csv"))
print(f"Score moyen ({BENCHMARK_LABEL}) : {df['score'].mean():.4f}")
print(f"Nb tâches   : {len(df)}")

# Par catégorie
categories = {
    "Classification" : ["MTOPDomainClassification", "MTOPIntentClassification", "MassiveScenarioClassification", "MassiveIntentClassification", "MasakhaNEWSClassification", "AmazonReviewsClassification"],
    "STS"            : ["STSBenchmarkMultilingualSTS", "SICKFr", "STS22"],
    "Reranking"      : ["SyntecReranking", "AlloprofReranking"],
    "PairClass."     : ["PawsXPairClassification"],
    "Clustering"     : ["AlloProfClusteringP2P", "AlloProfClusteringS2S", "MLSUMClusteringP2P.v2", "MLSUMClusteringS2S.v2", "MasakhaNEWSClusteringS2S", "MasakhaNEWSClusteringP2P", "HALClusteringS2S.v2"],
    "Retrieval"      : ["SyntecRetrieval", "XPQARetrieval", "AlloprofRetrieval", "MintakaRetrieval"],
    "Summarization"  : ["SummEvalFr"],
}

for cat, tasks in categories.items():
    scores = df[df["task"].isin(tasks)]["score"]
    print(f"{cat:<15} : {scores.mean():.4f}  ({len(scores)} tâches)")

Score moyen (MTEB(fra)-safe v2 CPU-only) : 0.4696
Nb tâches   : 25
Classification  : 0.6301  (6 tâches)
STS             : 0.7411  (3 tâches)
Reranking       : 0.5260  (2 tâches)
PairClass.      : 0.5883  (1 tâches)
Clustering      : 0.3750  (7 tâches)
Retrieval       : 0.2541  (4 tâches)
Summarization   : 0.2921  (1 tâches)


In [ ]:
# Publication optionnelle (à exécuter seulement après validation MTEB) :
# finetuned_model.push_to_hub("AurelPx/multilingual-e5-base-miracl-fr-lora")